# R による統計分析

R の組み込みデータセットを使用した仮説検定、回帰分析、信頼区間の計算。

データのダウンロードやパッケージのインストールは不要です — ベース R のみを使用します。

## 1. 記述統計

In [ ]:
data(mtcars)
cat("Dataset: mtcars (", nrow(mtcars), "cars, ", ncol(mtcars), "variables)\n\n")
summary(mtcars[, c("mpg", "hp", "wt", "disp")])

## 2. 2標本 t 検定

マニュアル車はオートマ車よりも燃費（MPG）が良いか？

In [ ]:
auto <- mtcars$mpg[mtcars$am == 0]
manual <- mtcars$mpg[mtcars$am == 1]

cat("Automatic:", round(mean(auto), 1), "mpg (n =", length(auto), ")\n")
cat("Manual:   ", round(mean(manual), 1), "mpg (n =", length(manual), ")\n\n")

t_result <- t.test(manual, auto, alternative = "greater")
print(t_result)

cat("\nConclusion:",
    ifelse(t_result$p.value < 0.05,
           "Reject H0 — manual cars have significantly higher MPG",
           "Fail to reject H0"))

## 3. カイ二乗検定

気筒数（cylinder count）とトランスミッションの種類は独立しているか？

In [ ]:
tab <- table(Cylinders = mtcars$cyl, Transmission = mtcars$am)
colnames(tab) <- c("Automatic", "Manual")
print(tab)
cat("\n")
chisq.test(tab)

## 4. 重回帰分析

In [ ]:
model <- lm(mpg ~ wt + hp + am, data = mtcars)
summary(model)

## 5. 回帰診断

In [ ]:
par(mfrow = c(2, 2))
plot(model)

## 6. 信頼区間

In [ ]:
ci <- confint(model, level = 0.95)
cat("95% Confidence Intervals:\n")
print(round(ci, 4))

In [ ]:
coefs <- coef(model)[-1]
ci_vals <- ci[-1, ]
n <- length(coefs)

par(mfrow = c(1, 1), mar = c(5, 8, 4, 2))
plot(coefs, 1:n, xlim = range(ci_vals),
     pch = 19, col = "#58a6ff", cex = 1.5,
     yaxt = "n", xlab = "Estimate", ylab = "",
     main = "95% Confidence Intervals for Coefficients")
axis(2, at = 1:n, labels = names(coefs), las = 1)
segments(ci_vals[, 1], 1:n, ci_vals[, 2], 1:n,
         lwd = 3, col = "#58a6ff")
abline(v = 0, lty = 2, col = "#f85149", lwd = 1.5)

## 7. 一元配置分散分析（One-Way ANOVA）

気筒数の違いによって燃費（MPG）に有意な差があるか？

In [ ]:
anova_model <- aov(mpg ~ factor(cyl), data = mtcars)
summary(anova_model)
cat("\nTukey HSD Post-hoc Comparisons:\n")
TukeyHSD(anova_model)

In [ ]:
boxplot(mpg ~ cyl, data = mtcars,
        main = "MPG by Cylinder Count",
        xlab = "Cylinders", ylab = "Miles per Gallon",
        col = c("#58a6ff", "#a371f7", "#f85149"))

## まとめ

- **ウェルチの t 検定**: 調整なしの片側比較において、マニュアル車の方が平均燃費（MPG）が高いという結果が得られます
- **カイ二乗検定**: 分割表は関連性を示唆していますが、期待度数の小ささにより近似精度の警告が発生するため、この結果の解釈には注意が必要です
- **回帰分析**: 変数調整後、車両重量（weight）と馬力（horsepower）が有意な負の予測因子となります。このモデルにおいてトランスミッションの種類は有意ではありません
- **ANOVA**: 4気筒、6気筒、8気筒のグループ間で燃費に有意な差があります。テューキーの多重比較（Tukey）の結果により、どのペア間に差があるかが特定されます